In [1]:
import sys
import pandas as pd
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor # <-- EL NUEVO CEREBRO
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

# =============================================================================
# 1. CONECTAR FUNCIONES DE PROCESADO
# =============================================================================
ruta_hugo = Path("../../Hugo").resolve()
if str(ruta_hugo) not in sys.path:
    sys.path.insert(0, str(ruta_hugo))

try:
    from channel_estimator import compute_channel_matrix_from_iq_paths
    from channel_features_complete import extract_channel_matrix_features
except ImportError:
    print("❌ ERROR: No se han podido importar las funciones de Hugo.")
    exit()

RUTA_YAML = Path("../../data/Modulator.yaml")
variables_clave = ['doppler_variance_energy', 'dH_dt_mean', 'svd_sigma_ratio']

# =============================================================================
# 2. CARGAR Y FUSIONAR BBDD ANTIGUAS (ENTRENAMIENTO)
# =============================================================================
print("🧠 1. Cargando y fusionando BBDD 1 y BBDD 2 (Entrenamiento)...")

ruta_bbdd_1 = '../dataset_features_temperatura.csv'
ruta_bbdd_2 = '../../data/dataset_features_temperatura.csv'

try:
    df_1 = pd.read_csv(ruta_bbdd_1)
    df_2 = pd.read_csv(ruta_bbdd_2)
    
    # Unimos ambas bases de datos masivas
    df_train = pd.concat([df_1, df_2], ignore_index=True)
    
    # Limpiamos por si hay algún hueco vacío en las variables que nos interesan
    df_train = df_train.dropna(subset=variables_clave + ['temperature'])
    print(f"✅ BBDD Histórica lista: {len(df_train)} muestras cargadas para entrenar.")
except Exception as e:
    print(f"❌ Error al cargar los CSV antiguos: {e}")
    exit()

X_train_bruto = df_train[variables_clave]
y_train = df_train['temperature']

# =============================================================================
# 3. EXTRAER DATOS EN VIVO PARA TEST (TESTEO)
# =============================================================================
print("\n📡 2. Extrayendo archivos .bin para usarlos como Test Ciego...")

materiales = ['carton', 'cristal', 'plastico']
condiciones = {'frio': 15.0, 'templado': 40.0, 'caliente': 90.0}
muestras = ['1', '2']

datos_test = []

for material in materiales:
    ruta_base = Path(f"../../data/{material}")
    for cond_str, temp_val in condiciones.items():
        for m in muestras:
            tx_path = ruta_base / cond_str / f"iq_tx_{m}.bin"
            rx_path = ruta_base / cond_str / f"iq_rx_{m}.bin"
            
            if not tx_path.exists() or not rx_path.exists():
                continue
                
            try:
                H = compute_channel_matrix_from_iq_paths(
                    tx_path=tx_path, rx_path=rx_path, yaml_path=RUTA_YAML,
                    fftshift=False, normalize_fft=False, trim_to_complete_frames=True,
                    output_order="mk", verbose=False
                )
                features = extract_channel_matrix_features(H, input_order="mk")
                
                fila = {var: features[var] for var in variables_clave if var in features}
                fila['temperature'] = temp_val
                fila['nombre'] = f"[{material.upper()}] {cond_str.capitalize()} - M. {m}"
                
                datos_test.append(fila)
            except Exception:
                pass # Ignoramos errores individuales para mantener la consola limpia

df_test = pd.DataFrame(datos_test)
X_test_bruto = df_test[variables_clave]
y_test = df_test['temperature']
nombres_test = df_test['nombre']

print(f"✅ Test listo: {len(df_test)} muestras extraídas.\n")

# =============================================================================
# 4. CONFIGURACIÓN DE GRADIENT BOOSTING (ESCALADO + ENTRENAMIENTO)
# =============================================================================
print("=" * 60)
print("⚙️ 3. Entrenando HistGradientBoosting (La bestia para datos ruidosos...)")
print("=" * 60)

# Aunque los árboles no lo exigen estrictamente, escalar ayuda a la estabilidad
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bruto)
X_test_scaled = scaler.transform(X_test_bruto)

# Configuramos Gradient Boosting. 
# 500 árboles, aprendiendo poco a poco (0.05), con penalización (L2) para no sobreajustar el ruido
modelo_gb = HistGradientBoostingRegressor(
    max_iter=500, 
    learning_rate=0.05, 
    l2_regularization=1.0, 
    random_state=42
)

modelo_gb.fit(X_train_scaled, y_train)

# =============================================================================
# 5. PREDICCIÓN Y RESULTADOS FINALES
# =============================================================================
predicciones = modelo_gb.predict(X_test_scaled)

print("\n🎯 RESULTADOS FINALES DE LA PREDICCIÓN (ENTRENADO CON HISTÓRICO):\n")

for nombre, temp_real, temp_pred in zip(nombres_test, y_test, predicciones):
    error = abs(temp_real - temp_pred)
    print(f"🌡️ {nombre.ljust(30)} -> Real: {temp_real:4.1f} ºC | IA: {temp_pred:4.1f} ºC | Error: {error:4.1f} ºC")

print("-" * 60)
mae = mean_absolute_error(y_test, predicciones)
r2 = r2_score(y_test, predicciones)

print(f"📉 Error Medio Absoluto (MAE): {mae:.2f} ºC")
print(f"📈 Puntuación R²: {r2:.2f} (1.0 es la perfección)")
print("=" * 60)

🧠 1. Cargando y fusionando BBDD 1 y BBDD 2 (Entrenamiento)...
✅ BBDD Histórica lista: 58238 muestras cargadas para entrenar.

📡 2. Extrayendo archivos .bin para usarlos como Test Ciego...
✅ Test listo: 18 muestras extraídas.

⚙️ 3. Entrenando HistGradientBoosting (La bestia para datos ruidosos...)

🎯 RESULTADOS FINALES DE LA PREDICCIÓN (ENTRENADO CON HISTÓRICO):

🌡️ [CARTON] Frio - M. 1           -> Real: 15.0 ºC | IA: 50.0 ºC | Error: 35.0 ºC
🌡️ [CARTON] Frio - M. 2           -> Real: 15.0 ºC | IA: 45.3 ºC | Error: 30.3 ºC
🌡️ [CARTON] Templado - M. 1       -> Real: 40.0 ºC | IA: 49.4 ºC | Error:  9.4 ºC
🌡️ [CARTON] Templado - M. 2       -> Real: 40.0 ºC | IA: 44.1 ºC | Error:  4.1 ºC
🌡️ [CARTON] Caliente - M. 1       -> Real: 90.0 ºC | IA: 49.8 ºC | Error: 40.2 ºC
🌡️ [CARTON] Caliente - M. 2       -> Real: 90.0 ºC | IA: 49.8 ºC | Error: 40.2 ºC
🌡️ [CRISTAL] Frio - M. 1          -> Real: 15.0 ºC | IA: 50.5 ºC | Error: 35.5 ºC
🌡️ [CRISTAL] Frio - M. 2          -> Real: 15.0 ºC | IA: 46.